# Simple and Weighted Forecast Combination

**Forecast combination** is one of the most robust strategies in time series forecasting.
The seminal paper by **Bates & Granger (1969)** showed that combining two forecasts
can reduce prediction error, even when one model is clearly inferior.

> *"The combination of forecasts is a simple, practical, and effective approach
> to improving forecast accuracy."* -- Bates & Granger (1969)

The **forecast combination puzzle** (Stock & Watson, 2004) refers to the empirical finding
that simple averages of forecasts often outperform more sophisticated weighting schemes.
This notebook explores simple and weighted combination methods to understand why.

**Topics covered:**
- Simple average combination
- Inverse MSE weighting
- Trimmed mean combination
- Performance comparison across methods

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.combination import SimpleCombiner, WeightedCombiner
from forecastbox.core.forecast import Forecast
from forecastbox.metrics import mae, rmse

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
np.random.seed(42)

## 1. Why Combine Forecasts?

Forecast combination works for two main reasons:

1. **Diversification**: Different models capture different patterns in the data.
   By combining them, we hedge against any single model being wrong.

2. **Variance reduction**: Even if all models are unbiased, their errors are
   imperfectly correlated. The combined forecast has lower variance than
   any individual forecast (analogous to portfolio diversification in finance).

Mathematically, for K models with equal weights:

$$\text{Var}(\bar{f}) = \frac{1}{K^2}\sum_{i=1}^{K}\sum_{j=1}^{K}\text{Cov}(e_i, e_j) \leq \frac{1}{K}\bar{\sigma}^2$$

The combined variance is always less than or equal to the average individual variance,
with equality only when all errors are perfectly correlated.

In [ ]:
# Load inflation forecasts dataset
df = pd.read_csv("../data/inflation_forecasts.csv", parse_dates=["date"])
print(f"Dataset shape: {df.shape}")
print(f"Period: {df['date'].min()} to {df['date'].max()}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Visualize individual forecasts vs actual
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df["date"], df["actual"], "k-", linewidth=2, label="Actual")
for col in ["fc_arima", "fc_ets", "fc_var", "fc_naive", "fc_drift"]:
    ax.plot(df["date"], df[col], "--", alpha=0.6, label=col.replace("fc_", "").upper())
ax.set_title("Individual Forecasts vs Actual Inflation")
ax.set_xlabel("Date")
ax.set_ylabel("Inflation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Simple Average

The simplest combination method assigns **equal weights** to all models:

$$\hat{y}_{c,t} = \frac{1}{K}\sum_{k=1}^{K}\hat{y}_{k,t}$$

Despite its simplicity, the simple average is remarkably hard to beat:
- It requires no estimation, so there is **no estimation error** in the weights.
- When the true optimal weights are uncertain, the simple average is a robust default.
- Stock & Watson (2004) showed it often outperforms estimated-weight methods in macro forecasting.

In [ ]:
# Split into train (first 80 obs) and test (last 40 obs)
n_train = 80
train_df = df.iloc[:n_train]
test_df = df.iloc[n_train:]

actual_train = train_df["actual"].values
actual_test = test_df["actual"].values

model_cols = ["fc_arima", "fc_ets", "fc_var", "fc_naive", "fc_drift"]
model_names = [c.replace("fc_", "").upper() for c in model_cols]

# Prepare training arrays for fit()
forecasts_train = [train_df[col].values for col in model_cols]

# Prepare Forecast objects for combine()
forecasts_test = [
    Forecast(point=test_df[col].values, model_name=name)
    for col, name in zip(model_cols, model_names)
]

# Simple Average combination
simple_combiner = SimpleCombiner(method="mean")
simple_combiner.fit(forecasts_train, actual_train)  # no-op for SimpleCombiner
fc_simple_avg = simple_combiner.combine(forecasts_test)

print("Simple Average weights:", simple_combiner.weights_)
print(f"MAE (Simple Avg): {mae(actual_test, fc_simple_avg.point):.4f}")
print(f"RMSE (Simple Avg): {rmse(actual_test, fc_simple_avg.point):.4f}")

# Compare with best individual model
print("\nIndividual model performance (test set):")
for name, col in zip(model_names, model_cols):
    print(f"  {name:8s}  MAE={mae(actual_test, test_df[col].values):.4f}  "
          f"RMSE={rmse(actual_test, test_df[col].values):.4f}")

## 3. Inverse MSE Weighting

Instead of equal weights, we can assign higher weights to models with **lower historical error**.
Inverse MSE weighting sets:

$$w_k = \frac{1/\text{MSE}_k}{\sum_{j=1}^{K} 1/\text{MSE}_j}$$

This is a natural performance-based weighting scheme:
- Models with lower MSE receive higher weights.
- The weights are always positive and sum to 1.
- It is equivalent to the **Bates-Granger (1969)** optimal weights when errors are uncorrelated.

In [ ]:
# Inverse MSE weighting
weighted_combiner = WeightedCombiner(method="inverse_mse")
weighted_combiner.fit(forecasts_train, actual_train)
fc_weighted = weighted_combiner.combine(forecasts_test)

# Display weights
print("Inverse MSE Weights:")
for name, w in zip(model_names, weighted_combiner.weights_):
    print(f"  {name:8s}: {w:.4f}")

print(f"\nTraining MSE per model:")
for name, m in zip(model_names, weighted_combiner.mse_):
    print(f"  {name:8s}: {m:.6f}")

print(f"\nMAE (Inverse MSE): {mae(actual_test, fc_weighted.point):.4f}")
print(f"RMSE (Inverse MSE): {rmse(actual_test, fc_weighted.point):.4f}")

# Visualize weights
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(model_names, weighted_combiner.weights_, color="steelblue")
ax.axhline(y=1/len(model_names), color="red", linestyle="--", label="Equal weight")
ax.set_ylabel("Weight")
ax.set_title("Inverse MSE Weights")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4. Trimmed Mean

The **trimmed mean** removes extreme forecasts before averaging.
This is useful when some models produce outlier predictions.

With a trim fraction of $\alpha$, we discard the lowest $\alpha$ and highest $\alpha$
fraction of forecasts at each time point, then average the remaining ones.

This is equivalent to removing the 2 worst models (for 5 models with ~20% trim)
and averaging the remaining 3.

In [ ]:
# Trimmed mean: remove the 2 most extreme forecasts at each time point
# With 5 models and trim_fraction=0.2, we trim 1 from each tail
trimmed_combiner = SimpleCombiner(method="trimmed", trim_fraction=0.2)
trimmed_combiner.fit(forecasts_train, actual_train)
fc_trimmed = trimmed_combiner.combine(forecasts_test)

print(f"Trimmed Mean (trim_fraction=0.2):")
print(f"  MAE:  {mae(actual_test, fc_trimmed.point):.4f}")
print(f"  RMSE: {rmse(actual_test, fc_trimmed.point):.4f}")

# Also try median combination
median_combiner = SimpleCombiner(method="median")
median_combiner.fit(forecasts_train, actual_train)
fc_median = median_combiner.combine(forecasts_test)

print(f"\nMedian Combination:")
print(f"  MAE:  {mae(actual_test, fc_median.point):.4f}")
print(f"  RMSE: {rmse(actual_test, fc_median.point):.4f}")

# Plot combined forecasts vs actual
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_df["date"], actual_test, "k-", linewidth=2, label="Actual")
ax.plot(test_df["date"], fc_simple_avg.point, "b--", label="Simple Avg")
ax.plot(test_df["date"], fc_trimmed.point, "r--", label="Trimmed Mean")
ax.plot(test_df["date"], fc_median.point, "g--", label="Median")
ax.set_title("Combined Forecasts vs Actual (Test Period)")
ax.set_xlabel("Date")
ax.set_ylabel("Inflation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Performance Comparison

Let's create a comprehensive comparison table of all individual models
and combination methods using MAE and RMSE.

In [ ]:
# Build comparison table
results = []

# Individual models
for name, col in zip(model_names, model_cols):
    pred = test_df[col].values
    results.append({
        "Method": name,
        "Type": "Individual",
        "MAE": mae(actual_test, pred),
        "RMSE": rmse(actual_test, pred),
    })

# Combination methods
combinations = {
    "Simple Average": fc_simple_avg,
    "Inverse MSE": fc_weighted,
    "Trimmed Mean": fc_trimmed,
    "Median": fc_median,
}

for name, fc in combinations.items():
    results.append({
        "Method": name,
        "Type": "Combination",
        "MAE": mae(actual_test, fc.point),
        "RMSE": rmse(actual_test, fc.point),
    })

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values("RMSE")
print("Performance Comparison (sorted by RMSE)")
print("=" * 55)
comparison_df.round(4)

## Exercise 1: Apply weighted combination to M4 sample series

Load `m4_sample.csv`, pick one series (e.g., `M4_001`), and apply both
simple average and inverse MSE weighting. Compare results.

In [ ]:
# TODO: Exercise 1
# 1. Load m4_sample.csv
# 2. Filter for series_id == "M4_001"
# 3. Split into train/test (80/20)
# 4. Apply SimpleCombiner(method='mean') and WeightedCombiner(method='inverse_mse')
# 5. Compare MAE and RMSE

## Exercise 2: Find optimal number of models to trim

Test different `trim_fraction` values from 0.0 to 0.4 and plot the
resulting RMSE. What is the optimal trim fraction for this dataset?

In [ ]:
# TODO: Exercise 2
# 1. Loop over trim_fraction in [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
# 2. For each, create SimpleCombiner(method='trimmed', trim_fraction=...)
# 3. Compute RMSE on the test set
# 4. Plot trim_fraction vs RMSE
# 5. Identify the optimal trim_fraction